# Statistiques descriptives — Base GDELT GKG (2015 → aujourd'hui)

Ce notebook explore la base Parquet hebdomadaire produite par `gdelt_weekly_pipeline.py`.

**Rappel du schéma de chaque fichier `gdelt_YYYY-MM.parquet` :**

| Colonne | Description |
|---|---|
| `GKGRECORDID` | identifiant unique de l'enregistrement GKG |
| `DATE` | horodatage GDELT au format `YYYYMMDDHHMMSS` (string) |
| `SourceCollectionIdentifier` | canal de collecte GDELT (1=WEB, 2=CITATIONONLY, 3=CORE, 4=DTIC, 5=JSTOR, 6=NONTEXTUALSOURCE — d'après la documentation GDELT) |
| `DocumentIdentifier` | URL de l'article source |
| `EnhancedThemes` | thèmes GKG (format `THEME,offset;THEME,offset;...`) |
| `EnhancedLocations` | lieux géolocalisés détectés |
| `Persons` | personnes citées (séparées par `;`) |
| `Organizations` | organisations citées (séparées par `;`) |
| `TranslationInfo` | infos de traduction (si article non anglophone) |
| `Tone` | tonalité moyenne de l'article (échelle GDELT, -100 à +100) |
| `WordCount` | nombre de mots approximatif de l'article |
| `translated` | 1 si l'article provient de la master list translingue, 0 sinon |
| `SourceCommonName_ID` | identifiant entier du média, à mapper via `gdelt_sources_mapping.json` |

**DuckDB** : interroger directement les fichiers Parquet sans tout charger en RAM 
    
**pandas/matplotlib/seaborn** : agrégations plus fines et visualisations


## 1. Installation & imports

In [2]:
import os
import glob
import json
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Configuration & inventaire des fichiers

In [3]:
DATA_DIR = Path("./gdelt_parquet_db")
SOURCE_MAP_PATH = Path("./gdelt_sources_mapping.json")

parquet_files = sorted(glob.glob(str(DATA_DIR / "gdelt_*.parquet")))
print(f"{len(parquet_files)} fichiers parquet (mois) trouvés.")

print("Premier fichier :", os.path.basename(parquet_files[0]))
print("Dernier fichier :", os.path.basename(parquet_files[-1]))
total_size_gb = sum(os.path.getsize(f) for f in parquet_files) / (1024**3)
print(f"Taille totale sur disque : {total_size_gb:.2f} Go")


137 fichiers parquet (mois) trouvés.
Premier fichier : gdelt_2015-02.parquet
Dernier fichier : gdelt_2026-06.parquet
Taille totale sur disque : 414.49 Go


## 3. Connexion DuckDB et vue unifiée sur l'ensemble des semaines

In [4]:
con = duckdb.connect()

glob_pattern = str(DATA_DIR / "gdelt_*.parquet")
con.execute(f"""
    CREATE OR REPLACE VIEW gkg AS
    SELECT * FROM read_parquet('{glob_pattern}')
""")

n_rows = con.execute("SELECT COUNT(*) FROM gkg").fetchone()[0]
print(f"Nombre total de lignes GKG : {n_rows:,}")


Nombre total de lignes GKG : 1,580,905,613


In [12]:
import json
import pandas as pd
from pathlib import Path

# 1. Définir le chemin vers ton fichier JSON (vérifie que c'est le bon chemin)
SOURCE_MAP_PATH = Path("./gdelt_sources_mapping.json")

# 2. Charger le dictionnaire depuis le fichier JSON
with open(SOURCE_MAP_PATH, "r", encoding="utf-8") as f:
    source_map = json.load(f)

# 3. Convertir le dictionnaire en DataFrame Pandas
id_to_source = source_map["id_to_source"]
src_df = pd.DataFrame({
    "SourceCommonName_ID": [int(k) for k in id_to_source.keys()],
    "SourceCommonName": list(id_to_source.values()),
})

# 4. Enregistrer le DataFrame comme une table virtuelle dans DuckDB
con.register("src_map", src_df)

print("✅ Fichier JSON chargé et table 'src_map' enregistrée dans DuckDB !")

✅ Fichier JSON chargé et table 'src_map' enregistrée dans DuckDB !


In [5]:
con.execute(r"""
    CREATE OR REPLACE VIEW gkg_clean AS
    SELECT *
    FROM gkg
    WHERE regexp_matches(CAST(DATE AS VARCHAR), '^\d{14}$')
      AND GKGRECORDID != '20210925181500-T1111'
""")

print("Le filtre est en place : la ligne '20210925181500-T1111' est retirée")

Le filtre est en place : la ligne '20210925181500-T1111' est retirée


In [6]:
con.execute("DESCRIBE gkg_clean").df()

,column_name,column_type,null,key,default,extra
0,GKGRECORDID,VARCHAR,YES,None,None,None
1,DATE,VARCHAR,YES,None,None,None
2,SourceCollectionIdentifier,TINYINT,YES,None,None,None
3,SourceCommonName_ID,INTEGER,YES,None,None,None
4,DocumentIdentifier,VARCHAR,YES,None,None,None
5,EnhancedThemes,VARCHAR,YES,None,None,None
6,EnhancedLocations,VARCHAR,YES,None,None,None
7,Persons,VARCHAR,YES,None,None,None
8,Organizations,VARCHAR,YES,None,None,None
9,TranslationInfo,VARCHAR,YES,None,None,None


In [ ]:
audit_query = """
-- =========================================================
-- 1. VOLUME ET UNICITÉ (Clés primaires)
-- =========================================================
SELECT '1. Volume & Unicité' AS Categorie, 'Nombre total d''articles' AS Indicateur, CAST(COUNT(*) AS VARCHAR) AS Valeur, 'ℹ️' AS Statut FROM gkg_clean
UNION ALL
SELECT '1. Volume & Unicité', 'Doublons sur GKGRECORDID', CAST(COUNT(*) - COUNT(DISTINCT GKGRECORDID) AS VARCHAR), CASE WHEN COUNT(*) = COUNT(DISTINCT GKGRECORDID) THEN '✅' ELSE '⚠️' END FROM gkg_clean

-- =========================================================
-- 2. CONFORMITÉ DES FORMATS ET RÈGLES MÉTIER
-- =========================================================
UNION ALL
SELECT '2. Conformité', 'URLs invalides (pas HTTP) sur Source=1', CAST(COUNT(*) AS VARCHAR), CASE WHEN COUNT(*) = 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean WHERE SourceCollectionIdentifier = 1 AND DocumentIdentifier NOT ILIKE 'http%'
UNION ALL
SELECT '2. Conformité', 'IsTranslingual (valeurs hors 0/1)', CAST(COUNT(*) AS VARCHAR), CASE WHEN COUNT(*) = 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean WHERE IsTranslingual NOT IN (0, 1)
UNION ALL
SELECT '2. Conformité', 'SourceCollectionIdentifier (hors 1 et 2)', CAST(COUNT(*) AS VARCHAR), CASE WHEN COUNT(*) = 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean WHERE SourceCollectionIdentifier NOT IN (1, 2)

-- =========================================================
-- 3. COMPLÉTUDE (% de valeurs nulles ou vides)
-- =========================================================
UNION ALL
SELECT '3. Complétude (Critique)', 'GKGRECORDID / DATE / URL (% Vides)', CAST(ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM gkg_clean), 4) AS VARCHAR) || ' %', CASE WHEN COUNT(*) = 0 THEN '✅' ELSE '⚠️' END FROM gkg_clean WHERE GKGRECORDID IS NULL OR DATE IS NULL OR DocumentIdentifier IS NULL OR DocumentIdentifier = ''
UNION ALL
SELECT '3. Complétude (Information)', 'EnhancedThemes (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN EnhancedThemes IS NULL OR EnhancedThemes = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean
UNION ALL
SELECT '3. Complétude (Information)', 'EnhancedLocations (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN EnhancedLocations IS NULL OR EnhancedLocations = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean
UNION ALL
SELECT '3. Complétude (Information)', 'Persons (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN Persons IS NULL OR Persons = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean
UNION ALL
SELECT '3. Complétude (Information)', 'Organizations (% Vides)', CAST(ROUND(100.0 * SUM(CASE WHEN Organizations IS NULL OR Organizations = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean
UNION ALL
SELECT '3. Complétude (Information)', 'TranslationInfo (% Vides - Normal)', CAST(ROUND(100.0 * SUM(CASE WHEN TranslationInfo IS NULL OR TranslationInfo = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS VARCHAR) || ' %', 'ℹ️' FROM gkg_clean

-- =========================================================
-- 4. PROFILING DES VARIABLES NUMÉRIQUES (Avec Distribution)
-- =========================================================
UNION ALL
SELECT 
    '4. Profiling Numérique', 
    'Tone (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(ROUND(MIN(Tone), 1) AS VARCHAR) || ' | ' || 
    CAST(ROUND(APPROX_QUANTILE(Tone, 0.25), 1) AS VARCHAR) || ' | ' || 
    CAST(ROUND(APPROX_QUANTILE(Tone, 0.50), 1) AS VARCHAR) || ' | ' || 
    CAST(ROUND(AVG(Tone), 2) AS VARCHAR) || ' | ' || 
    CAST(ROUND(APPROX_QUANTILE(Tone, 0.75), 1) AS VARCHAR) || ' | ' || 
    CAST(ROUND(MAX(Tone), 1) AS VARCHAR), 
    CASE WHEN MIN(Tone) >= -100 AND MAX(Tone) <= 100 THEN '✅' ELSE '⚠️' END 
FROM gkg_clean

UNION ALL
SELECT 
    '4. Profiling Numérique', 
    'WordCount (Min | Q1 | Médiane | Moyenne | Q3 | Max)', 
    CAST(MIN(WordCount) AS VARCHAR) || ' | ' || 
    CAST(CAST(APPROX_QUANTILE(WordCount, 0.25) AS INTEGER) AS VARCHAR) || ' | ' || 
    CAST(CAST(APPROX_QUANTILE(WordCount, 0.50) AS INTEGER) AS VARCHAR) || ' | ' || 
    CAST(ROUND(AVG(WordCount), 0) AS VARCHAR) || ' | ' || 
    CAST(CAST(APPROX_QUANTILE(WordCount, 0.75) AS INTEGER) AS VARCHAR) || ' | ' || 
    CAST(MAX(WordCount) AS VARCHAR), 
    CASE WHEN MIN(WordCount) >= 0 THEN '✅' ELSE '⚠️' END 
FROM gkg_clean

-- =========================================================
-- 5. RICHESSE SÉMANTIQUE (Nombre moyen d'entités par article)
-- =========================================================
UNION ALL
SELECT '5. Richesse Sémantique', 'Nb moyen de Thèmes par article', CAST(ROUND(AVG(LENGTH(EnhancedThemes) - LENGTH(REPLACE(EnhancedThemes, ';', '')) + 1), 1) AS VARCHAR), 'ℹ️' FROM gkg_clean WHERE EnhancedThemes IS NOT NULL AND EnhancedThemes != ''
UNION ALL
SELECT '5. Richesse Sémantique', 'Nb moyen de Personnes par article', CAST(ROUND(AVG(LENGTH(Persons) - LENGTH(REPLACE(Persons, ';', '')) + 1), 1) AS VARCHAR), 'ℹ️' FROM gkg_clean WHERE Persons IS NOT NULL AND Persons != ''
UNION ALL
SELECT '5. Richesse Sémantique', 'Nb moyen d''Organisations par article', CAST(ROUND(AVG(LENGTH(Organizations) - LENGTH(REPLACE(Organizations, ';', '')) + 1), 1) AS VARCHAR), 'ℹ️' FROM gkg_clean WHERE Organizations IS NOT NULL AND Organizations != ''

ORDER BY Categorie, Indicateur;
"""

# Exécution et affichage ajusté
dashboard_sante_v3 = con.execute(audit_query).df()

styled_dashboard = dashboard_sante_v3.style.set_properties(**{'text-align': 'left'}, subset=['Categorie', 'Indicateur', 'Valeur']) \
                                           .set_properties(**{'text-align': 'center'}, subset=['Statut']) \
                                           .hide(axis="index")
styled_dashboard

Categorie,Indicateur,Valeur,Statut
1. Volume & Unicité,Doublons sur GKGRECORDID,0,✅
1. Volume & Unicité,Nombre total d'articles,1580905612,ℹ️
2. Conformité,IsTranslingual (valeurs hors 0/1),0,✅
2. Conformité,SourceCollectionIdentifier (hors 1 et 2),0,✅
2. Conformité,URLs invalides (pas HTTP) sur Source=1,0,✅
3. Complétude (Critique),GKGRECORDID / DATE / URL (% Vides),0.0 %,✅
3. Complétude (Information),EnhancedLocations (% Vides),21.78 %,ℹ️
3. Complétude (Information),EnhancedThemes (% Vides),9.27 %,ℹ️
3. Complétude (Information),Organizations (% Vides),31.8 %,ℹ️
3. Complétude (Information),Persons (% Vides),43.56 %,ℹ️


In [14]:
con.execute("SELECT * FROM gkg LIMIT 5").df()

,GKGRECORDID,DATE,SourceCollectionIdentifier,SourceCommonName_ID,DocumentIdentifier,EnhancedThemes,EnhancedLocations,Persons,Organizations,TranslationInfo,IsTranslingual,Tone,WordCount
0,20150219071500-0,20150219071500,2,1277,"Tolo TV, Kabul/BBC Monitoring/(c) BBC","KILL,464;SEIGE,3001;CHECKPOINT,3001;UNREST_CHE...",1#Afghan#AF#AF##33#65#AF#841;1#Afghan#AF#AF##3...,sharif amiri,,,0,-4.64,499
1,20150219020000-0,20150219020000,1,683,http://www.hardenexpress.com.au/story/2893542/...,"CRIME_COMMON_ROBBERY,797;CRIME_COMMON_ROBBERY,...","4#Sydney, New South Wales, Australia#AS#AS02#1...",robert van winkle,,,0,-2.22,312
2,20150219020000-1,20150219020000,1,684,http://www.wsbt.com/wsbt-extras/wsbt-radio/213...,"TRAFFIC,986;DISCRIMINATION,569;ARREST,1012;TRI...",,darren wilson;michael brown,ferguson police department;national press club...,,0,-2.07,269
3,20150219020000-2,20150219020000,1,685,http://www.buffalonews.com/city-region/is-bull...,"GENERAL_GOVERNMENT,2276;ECON_INTEREST_RATES,22...",,,investech research,,0,-2.25,362
4,20150219020000-3,20150219020000,1,33,http://www.ad-hoc-news.de/92-year-old-wisconsi...,"TAX_WORLDLANGUAGES,2843;TAX_FNCACT,3153;TAX_FN...","3#University Park, Illinois, United States#US#...",scott walker;nittany lions;mike rowe,university park;club in manhattan;cnn;statoil;...,,0,-1.10,553


In [15]:
import pandas as pd
from IPython.display import display, HTML

con.execute("PRAGMA memory_limit='200GB'")

print("Tops on a uniform 10% sample (Hash Method)...")

# 2. Top 20 Themes
top_themes_sample = con.execute("""
    SELECT split_part(TRIM(theme_raw), ',', 1) AS Theme, COUNT(*) AS Citations
    FROM (
        SELECT UNNEST(string_split(EnhancedThemes, ';')) AS theme_raw
        FROM gkg_clean
        WHERE hash(GKGRECORDID) % 10 = 0
          AND EnhancedThemes IS NOT NULL AND EnhancedThemes != ''
    )
    WHERE TRIM(theme_raw) != ''
    GROUP BY Theme 
    ORDER BY Citations DESC 
    LIMIT 50
""").df()

# 3. Top 20 Persons
top_persons_sample = con.execute("""
    SELECT TRIM(person) AS Person, COUNT(*) AS Citations
    FROM (
        SELECT UNNEST(string_split(Persons, ';')) AS person
        FROM gkg_clean
        WHERE hash(GKGRECORDID) % 10 = 0
          AND Persons IS NOT NULL AND Persons != ''
    )
    WHERE TRIM(person) != ''
    GROUP BY Person 
    ORDER BY Citations DESC 
    LIMIT 50
""").df()

# 4. Top 20 Organizations
top_orgs_sample = con.execute("""
    SELECT TRIM(org) AS Organization, COUNT(*) AS Citations
    FROM (
        SELECT UNNEST(string_split(Organizations, ';')) AS org
        FROM gkg_clean
        WHERE hash(GKGRECORDID) % 10 = 0
          AND Organizations IS NOT NULL AND Organizations != ''
    )
    WHERE TRIM(org) != ''
    GROUP BY Organization 
    ORDER BY Citations DESC 
    LIMIT 50
""").df()

# 5. Top 20 Locations (Splitting on '#' to extract the readable name)
top_locs_sample = con.execute("""
    SELECT split_part(TRIM(loc), '#', 2) AS Location, COUNT(*) AS Citations
    FROM (
        SELECT UNNEST(string_split(EnhancedLocations, ';')) AS loc
        FROM gkg_clean
        WHERE hash(GKGRECORDID) % 10 = 0
          AND EnhancedLocations IS NOT NULL AND EnhancedLocations != ''
    )
    WHERE TRIM(loc) != '' AND split_part(TRIM(loc), '#', 2) != ''
    GROUP BY Location 
    ORDER BY Citations DESC 
    LIMIT 50
""").df()

# 6. Top 20 Sources (JOIN with src_map to get textual names)
top_sources_sample = con.execute("""
    SELECT m.SourceCommonName AS Source_Name, CAST(g.SourceCommonName_ID AS VARCHAR) AS Source_ID, COUNT(*) AS Articles
    FROM gkg_clean g
    JOIN src_map m ON g.SourceCommonName_ID = m.SourceCommonName_ID
    WHERE hash(g.GKGRECORDID) % 10 = 0
      AND g.SourceCommonName_ID IS NOT NULL
    GROUP BY m.SourceCommonName, g.SourceCommonName_ID
    ORDER BY Articles DESC 
    LIMIT 50
""").df()

print("Calculations complete, displaying dashboards:")

# 6. Fonction pour afficher les DataFrames côte à côte (Corrigée pour éviter le chevauchement)
def display_side_by_side(dfs, titles):
    # 'flex: 0 0 auto' empêche l'écrasement, les tableaux iront à la ligne si l'écran est trop petit
    html_str = '<div style="display: flex; flex-wrap: wrap; gap: 30px; align-items: flex-start;">'
    
    for df, title in zip(dfs, titles):
        # On ajoute des règles CSS pour que le texte long revienne à la ligne proprement
        styled_df = df.style.format(thousands=",") \
                            .hide(axis="index") \
                            .set_table_styles([
                                {'selector': 'td, th', 
                                 'props': [('max-width', '250px'), 
                                           ('white-space', 'normal'), 
                                           ('word-wrap', 'break-word')]}
                            ])
        
        html_str += f'<div style="flex: 0 0 auto; overflow-x: auto;">'
        html_str += f'<h3 style="text-align: center; margin-bottom: 10px;">{title}</h3>'
        html_str += styled_df.to_html()
        html_str += '</div>'
        
    html_str += '</div>'
    display(HTML(html_str))

# 7. Affichage final
display_side_by_side(
    [top_themes_sample, top_persons_sample, top_orgs_sample, top_locs_sample, top_sources_sample], 
    ['Themes', 'Persons', 'Organizations', 'Locations', 'Sources']
)

Tops on a uniform 10% sample (Hash Method)...
Calculations complete, displaying dashboards:


Theme,Citations
USPEC_POLITICS_GENERAL1,"89,934,962"
CRISISLEX_C07_SAFETY,"85,763,766"
CRISISLEX_CRISISLEXREC,"84,162,274"
LEADER,"83,259,282"
GENERAL_HEALTH,"79,043,577"
GENERAL_GOVERNMENT,"75,240,091"
UNGP_FORESTS_RIVERS_OCEANS,"70,794,236"
MANMADE_DISASTER_IMPLIED,"69,653,700"
EPU_ECONOMY_HISTORIC,"69,405,257"
EDUCATION,"69,083,807"


## 4. Describe des variables numériques (`Tone`, `WordCount`, `translated`)

In [22]:
trans_ratio = con.execute("""
    SELECT TranslationInfo,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM gkg_clean
    GROUP BY TranslationInfo
    ORDER BY pct DESC
    LIMIT 21
""").df()
print("Répartition articles natifs (0) vs traduits (1) :")
trans_ratio

Répartition articles natifs (0) vs traduits (1) :


,TranslationInfo,n,pct
0,,680204580,43.03
1,srclc:spa;eng:GT-SPA 1.0,112451365,7.11
2,srclc:zho;eng:GT-ZHO 1.0,87086707,5.51
3,srclc:deu;eng:GT-DEU 1.0,69374677,4.39
4,srclc:spa;eng:Moses 2.1.1 / MosesCore Europarl...,61091860,3.86
5,srclc:ita;eng:GT-ITA 1.0,60402793,3.82
6,srclc:rus;eng:GT-RUS 1.0,60194165,3.81
7,srclc:fra;eng:GT-FRA 1.0,43372936,2.74
8,srclc:tur;eng:GT-TUR 1.0,42175712,2.67
9,srclc:ara;eng:GT-ARA 1.0,29748115,1.88


In [23]:
collection_dist = con.execute("""
    SELECT SourceCollectionIdentifier, COUNT(*) AS n
    FROM gkg
    GROUP BY SourceCollectionIdentifier
    ORDER BY n DESC
""").df()
collection_dist


,SourceCollectionIdentifier,n
0,1,1580555920
1,2,349693


## 5. Journaux (sources) les plus représentés

In [ ]:
with open(SOURCE_MAP_PATH, "r", encoding="utf-8") as f:
    source_map = json.load(f)

id_to_source = source_map["id_to_source"]
src_df = pd.DataFrame({
    "SourceCommonName_ID": [int(k) for k in id_to_source.keys()],
    "SourceCommonName": list(id_to_source.values()),
})
print(f"{len(src_df):,} sources distinctes référencées dans le dictionnaire.")


In [ ]:
con.register("src_map", src_df)

top_sources = con.execute("""
    SELECT m.SourceCommonName, COUNT(*) AS n_articles
    FROM gkg g
    JOIN src_map m ON g.SourceCommonName_ID = m.SourceCommonName_ID
    GROUP BY m.SourceCommonName
    ORDER BY n_articles DESC
    LIMIT 30
""").df()
top_sources


In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(data=top_sources, y="SourceCommonName", x="n_articles", color="steelblue")
plt.title("Top 30 des sources (médias) les plus représentées")
plt.xlabel("Nombre d'articles")
plt.ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
n_distinct_sources_used = con.execute("""
    SELECT COUNT(DISTINCT SourceCommonName_ID) FROM gkg
""").fetchone()[0]
print(f"Nombre de médias distincts apparaissant réellement dans la base : {n_distinct_sources_used:,}")


## 6. Volume d'articles par jour

On reconstruit le jour calendaire à partir des 8 premiers caractères de `DATE` (`YYYYMMDD`).


In [ ]:
articles_per_day = con.execute("""
    SELECT STRPTIME(SUBSTR(DATE, 1, 8), '%Y%m%d')::DATE AS day,
           COUNT(*) AS n_articles
    FROM gkg
    GROUP BY day
    ORDER BY day
""").df()

articles_per_day["day"] = pd.to_datetime(articles_per_day["day"])
articles_per_day = articles_per_day.set_index("day")

print(f"Nombre de jours couverts par la collecte : {len(articles_per_day):,}")
print(f"Total d'articles                          : {articles_per_day['n_articles'].sum():,.0f}")
print(f"Moyenne d'articles / jour                 : {articles_per_day['n_articles'].mean():,.1f}")
print(f"Médiane d'articles / jour                 : {articles_per_day['n_articles'].median():,.1f}")
print(f"Écart-type / jour                         : {articles_per_day['n_articles'].std():,.1f}")
print(f"Min / jour                                : {articles_per_day['n_articles'].min():,.0f}")
print(f"Max / jour                                : {articles_per_day['n_articles'].max():,.0f}")


In [ ]:
# Détection de jours manquants dans la collecte (utile pour évaluer la qualité du pipeline)
full_range = pd.date_range(articles_per_day.index.min(), articles_per_day.index.max(), freq="D")
missing_days = full_range.difference(articles_per_day.index)

print(f"Jours sans aucune donnée sur la période : {len(missing_days)} / {len(full_range)}")
if len(missing_days) > 0:
    print("Premiers jours manquants :", list(missing_days[:15].strftime('%Y-%m-%d')))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
articles_per_day["n_articles"].plot(ax=ax, linewidth=0.7, color="steelblue")
articles_per_day["n_articles"].rolling(30, min_periods=1).mean().plot(
    ax=ax, linewidth=2, color="darkorange", label="Moyenne mobile 30 jours"
)
ax.set_title("Nombre d'articles GKG collectés par jour")
ax.set_ylabel("Articles / jour")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
monthly_volume = articles_per_day["n_articles"].resample("MS").sum()

fig, ax = plt.subplots(figsize=(14, 5))
monthly_volume.plot(ax=ax, marker="o", markersize=3, color="seagreen")
ax.set_title("Volume mensuel total d'articles GKG")
ax.set_ylabel("Articles / mois")
plt.tight_layout()
plt.show()


In [ ]:
weekday_avg = (
    articles_per_day.assign(weekday=articles_per_day.index.day_name())
    .groupby("weekday")["n_articles"].mean()
    .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
)

weekday_avg.plot(kind="bar", color="darkorange")
plt.title("Moyenne d'articles par jour de la semaine")
plt.ylabel("Articles (moyenne)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Distribution du `Tone` (tonalité)

Le `Tone` GDELT varie typiquement entre -10 (très négatif) et +10 (très positif), avec une masse centrée autour de 0.
On travaille ici sur un échantillon pour la visualisation (la base complète peut être trop volumineuse pour un histogramme direct).


In [ ]:
tone_sample = con.execute("SELECT Tone FROM gkg USING SAMPLE 500000 ROWS").df()

plt.figure()
sns.histplot(tone_sample["Tone"], bins=100, kde=True, color="slateblue")
plt.title("Distribution du Tone (échantillon de 500k articles)")
plt.xlabel("Tone GDELT")
plt.show()

tone_sample["Tone"].describe()


In [ ]:
tone_monthly = con.execute("""
    SELECT DATE_TRUNC('month', STRPTIME(SUBSTR(DATE, 1, 8), '%Y%m%d')::DATE) AS month,
           AVG(Tone) AS avg_tone,
           COUNT(*) AS n
    FROM gkg
    GROUP BY month
    ORDER BY month
""").df()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(tone_monthly["month"], tone_monthly["avg_tone"], color="firebrick")
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
ax.set_title("Tone moyen mensuel — proxy de sentiment médiatique global")
ax.set_ylabel("Tone moyen")
plt.tight_layout()
plt.show()


## 8. Distribution du `WordCount`

In [ ]:
wc_sample = con.execute("SELECT WordCount FROM gkg USING SAMPLE 500000 ROWS").df()
q99 = wc_sample["WordCount"].quantile(0.99)

plt.figure()
sns.histplot(wc_sample.loc[wc_sample["WordCount"] < q99, "WordCount"], bins=80, color="teal")
plt.title("Distribution du WordCount (99e percentile exclu, échantillon)")
plt.xlabel("Nombre de mots")
plt.show()

wc_sample["WordCount"].describe()


## 9. Valeurs manquantes / vides par colonne

In [ ]:
cols_to_check = [
    "GKGRECORDID", "DATE", "SourceCollectionIdentifier", "DocumentIdentifier",
    "EnhancedThemes", "EnhancedLocations", "Persons", "Organizations",
    "TranslationInfo", "Tone", "WordCount", "SourceCommonName_ID",
]

null_rows = []
for c in cols_to_check:
    n_null = con.execute(
        f"SELECT COUNT(*) FROM gkg WHERE {c} IS NULL OR CAST({c} AS VARCHAR) = ''"
    ).fetchone()[0]
    null_rows.append({"colonne": c, "pct_null_ou_vide": round(100 * n_null / n_rows, 2)})

pd.DataFrame(null_rows).sort_values("pct_null_ou_vide", ascending=False).reset_index(drop=True)


## 10. Thèmes, personnes et organisations les plus cités

Ces champs sont des listes concaténées (`;` entre entités). On travaille sur un **échantillon** pour rester rapide
— pour un traitement exhaustif (utile à la construction d'indicateurs), il faudra industrialiser cet "explode"
en dehors du notebook (Spark, DuckDB `UNNEST`, ou traitement par chunks).


In [ ]:
sample_df = con.execute("""
    SELECT EnhancedThemes, Persons, Organizations
    FROM gkg
    USING SAMPLE 200000 ROWS
""").df()

def top_n_from_semicolon_field(series, n=20, strip_offset=False):
    counter = {}
    for val in series.dropna():
        if not val:
            continue
        for item in val.split(";"):
            item = item.strip()
            if not item:
                continue
            if strip_offset:
                item = item.split(",")[0]
            counter[item] = counter.get(item, 0) + 1
    return pd.Series(counter, dtype="int64").sort_values(ascending=False).head(n)

top_themes = top_n_from_semicolon_field(sample_df["EnhancedThemes"], strip_offset=True)
top_persons = top_n_from_semicolon_field(sample_df["Persons"], strip_offset=False)
top_orgs = top_n_from_semicolon_field(sample_df["Organizations"], strip_offset=False)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

top_themes.sort_values().plot(kind="barh", ax=axes[0], color="seagreen")
axes[0].set_title("Top 20 thèmes GKG")

top_persons.sort_values().plot(kind="barh", ax=axes[1], color="indianred")
axes[1].set_title("Top 20 personnes citées")

top_orgs.sort_values().plot(kind="barh", ax=axes[2], color="goldenrod")
axes[2].set_title("Top 20 organisations citées")

plt.tight_layout()
plt.show()


In [ ]:
# Complexité moyenne d'un article : nombre d'entités détectées
sample_df["n_themes"] = sample_df["EnhancedThemes"].fillna("").apply(lambda x: len([i for i in x.split(";") if i]))
sample_df["n_persons"] = sample_df["Persons"].fillna("").apply(lambda x: len([i for i in x.split(";") if i]))
sample_df["n_orgs"] = sample_df["Organizations"].fillna("").apply(lambda x: len([i for i in x.split(";") if i]))

sample_df[["n_themes", "n_persons", "n_orgs"]].describe()


## 11. Corrélations entre variables numériques

In [ ]:
corr_sample = con.execute("SELECT Tone, WordCount FROM gkg USING SAMPLE 500000 ROWS").df()
corr = corr_sample.corr()

plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Corrélations entre variables numériques")
plt.tight_layout()
plt.show()


In [ ]:
con.close()
print("Connexion DuckDB fermée.")


## 12. Pistes pour construire des indicateurs à partir des GKG events

Quelques directions naturelles à partir de cette base, par ordre croissant de complexité :

- **Indices d'attention médiatique** : volume d'articles par jour filtré sur un thème (`EnhancedThemes` contient `ECON_*`, `CRISISLEX_*`, `ENV_*`...), normalisé par le volume total du jour pour neutraliser les variations de couverture globale.
- **Indices de sentiment thématique** : moyenne du `Tone` pondérée par `WordCount`, calculée séparément pour chaque thème ou pays cité — utile pour suivre l'évolution de la perception médiatique d'un sujet dans le temps.
- **Détection de pics / anomalies** : z-score glissant sur le volume journalier ou sur le `Tone` moyen journalier, pour repérer des ruptures (crises, événements majeurs).
- **Cartographie géographique** : `EnhancedLocations` contient latitude/longitude — on peut agréger par pays/région pour des cartes de chaleur d'attention médiatique.
- **Réseaux de co-occurrence** : construire un graphe (NetworkX) où les nœuds sont des `Persons`/`Organizations` apparaissant dans le même article, pour identifier des clusters d'acteurs liés à un sujet.
- **Décomposition temporelle** : STL ou décomposition saisonnière classique sur les séries de volume/tone pour séparer tendance, saisonnalité et résidu.
- **Biais médiatique cross-source** : comparer le `Tone` moyen par média (`SourceCommonName`) sur un même thème/période pour repérer des différences de traitement éditorial.

Pour l'industrialisation de ces traitements sur l'ensemble de la base (et non un échantillon), DuckDB propose `UNNEST` pour exploser nativement les colonnes `;`-séparées sans repasser par pandas, ce qui sera nettement plus rapide à l'échelle de plusieurs années de données.
